In [2]:
import pandas as pd 
import numpy as np
import tensorflow as tf 
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers,models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau
import keras_tuner as kt
import tensorboard

In [3]:
df_train = pd.read_csv(r'../data/selected_col/model_train.csv')
df_test = pd.read_csv(r'../data/selected_col/model_test.csv')
df_val = pd.read_csv(r'../data/selected_col/model_val.csv')

In [4]:
img_size = (224,224)
batch_size = 32
batch_size_64 = 64

In [5]:
data_augmentation_2 = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.2),
])

In [6]:
def preprocess_image2(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, img_size)
    return image, label

In [7]:
def preprocess_train2(image_path, label):
    image, label = preprocess_image2(image_path, label)
    image = data_augmentation_2(image)
    return image, label

In [8]:
def preprocess_test2(image_path, label):
    image, label = preprocess_image2(image_path, label)
    return image, label

In [43]:
train_dataset2 = tf.data.Dataset.from_tensor_slices(
    (
        df_train["path"].values,
        df_train["dx_encode"].values
    )
)

train_dataset2 = (
    train_dataset2
    .map(preprocess_train2, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

In [44]:
test_dataset2 = tf.data.Dataset.from_tensor_slices(
    (
        df_test["path"].values,
        df_test["dx_encode"].values
    )
)

test_dataset2 = (
    test_dataset2
    .map(preprocess_test2, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

In [45]:
val_dataset2 = tf.data.Dataset.from_tensor_slices(
    (
        df_val["path"].values,
        df_val["dx_encode"].values
    )
)

val_dataset2 = (
    val_dataset2
    .map(preprocess_test2, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)


In [40]:
train_dataset_64 = tf.data.Dataset.from_tensor_slices(
    (
        df_train["path"].values,
        df_train["dx_encode"].values
    )
)

train_dataset_64 = (
    train_dataset_64
    .map(preprocess_train2, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000)
    .batch(batch_size_64)
    .prefetch(tf.data.AUTOTUNE)
)

In [41]:
test_dataset_64 = tf.data.Dataset.from_tensor_slices(
    (
        df_test["path"].values,
        df_test["dx_encode"].values
    )
)

test_dataset_64 = (
    test_dataset_64
    .map(preprocess_test2, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size_64)
    .prefetch(tf.data.AUTOTUNE)
)

In [42]:
val_dataset_64 = tf.data.Dataset.from_tensor_slices(
    (
        df_val["path"].values,
        df_val["dx_encode"].values
    )
)

val_dataset_64 = (
    val_dataset_64
    .map(preprocess_test2, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size_64)
    .prefetch(tf.data.AUTOTUNE)
)

In [12]:
y_train = df_train["dx_encode"].values


class_weights = compute_class_weight(class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train)

# Convert to dictionary
class_weight_dict = dict(enumerate(class_weights))

class_weight_dict

{0: np.float64(4.372426699937617),
 1: np.float64(2.7813492063492062),
 2: np.float64(1.3020620471855842),
 3: np.float64(12.51607142857143),
 4: np.float64(1.2853475151292866),
 5: np.float64(0.2133572798392743),
 6: np.float64(10.113997113997113)}

In [13]:
'''CNN BASELINE WITH EARLY STOP'''

early_stop = EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True,verbose=1)

In [14]:
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

In [ ]:
#early stop

In [13]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

resnet_model = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),

    layers.Dropout(0.5),

    layers.Dense(7, activation="softmax")

])

resnet_model.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 17s 0us/step


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,850,887 (90.98 MB)

 Trainable params: 263,175 (1.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [14]:
resnet_model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [17]:
history = resnet_model.fit(train_dataset2,validation_data=val_dataset2,epochs=5,class_weight=class_weight_dict,callbacks=[early_stop])

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 374s 2s/step - accuracy: 0.4072 - loss: 1.6981 - val_accuracy: 0.4551 - val_loss: 1.3828
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 362s 2s/step - accuracy: 0.4597 - loss: 1.4178 - val_accuracy: 0.4917 - val_loss: 1.2945
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 364s 2s/step - accuracy: 0.4790 - loss: 1.3361 - val_accuracy: 0.5828 - val_loss: 1.0553
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 363s 2s/step - accuracy: 0.4928 - loss: 1.2900 - val_accuracy: 0.5496 - val_loss: 1.0767
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 368s 2s/step - accuracy: 0.5142 - loss: 1.2118 - val_accuracy: 0.5250 - val_loss: 1.1082
Restoring model weights from the end of the best epoch: 3.


In [18]:
resnet_model.evaluate(test_dataset2)

47/47 ━━━━━━━━━━━━━━━━━━━━ 61s 1s/step - accuracy: 0.5988 - loss: 1.0569


[1.056898832321167, 0.598802387714386]

In [19]:
resnet_model.save(r'../models/resnet_early_stop.keras')

In [ ]:
'''learning rate'''

In [20]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

resnet_model_lr = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),

    layers.Dropout(0.5),

    layers.Dense(7, activation="softmax")

])

resnet_model_lr.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,850,887 (90.98 MB)

 Trainable params: 263,175 (1.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [21]:
resnet_model_lr.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [22]:
history = resnet_model_lr.fit(train_dataset2,validation_data=val_dataset2,epochs=5,class_weight=class_weight_dict,callbacks=[early_stop,lr_scheduler])

Epoch 1/5


c:\Users\rizwa\.virtualenvs\week_7-vVhlwxiV\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 367s 2s/step - accuracy: 0.3789 - loss: 1.6990 - val_accuracy: 0.4637 - val_loss: 1.2802 - learning_rate: 0.0010
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 362s 2s/step - accuracy: 0.4608 - loss: 1.4156 - val_accuracy: 0.4291 - val_loss: 1.3877 - learning_rate: 0.0010
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 361s 2s/step - accuracy: 0.4835 - loss: 1.3149 - val_accuracy: 0.6214 - val_loss: 0.9862 - learning_rate: 0.0010
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 362s 2s/step - accuracy: 0.4962 - loss: 1.2619 - val_accuracy: 0.5130 - val_loss: 1.2480 - learning_rate: 0.0010
Epoch 5/5
219/220 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.5108 - loss: 1.2341
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
220/220 ━━━━━━━━━━━━━━━━━━━━ 361s 2s/step - accuracy: 0.5108 - loss: 1.2339 - val_accuracy: 0.5050 - val_loss: 1.1812 - learning_rate: 0.0010
Restoring model weights from the end of the best epoch: 3.


In [23]:
resnet_model_lr.evaluate(test_dataset2)

47/47 ━━━━━━━━━━━━━━━━━━━━ 64s 1s/step - accuracy: 0.6168 - loss: 0.9888


[0.9888402819633484, 0.6167664527893066]

In [24]:
resnet_model_lr.save(r'../models/resnet_lr.keras')

In [25]:
history_df =pd.DataFrame(history.history)

history_df.to_csv(r'../log/res_net_lr.csv',index=False)


In [ ]:
#hyper parameter

In [1]:
INPUT_SHAPE = (224,224,3)
NUM_CLASSES = 7

def build_resnet(hp):

    base_model = tf.keras.applications.ResNet50(
        include_top=False,
        weights="imagenet",
        input_shape=INPUT_SHAPE
    )

    base_model.trainable = False

    model = tf.keras.Sequential([

        base_model,

        tf.keras.layers.GlobalAveragePooling2D(),

        tf.keras.layers.Dense(
            hp.Choice(
                "dense_units",
                values=[128,256,512]
            ),
            activation="relu"
        ),

        tf.keras.layers.Dropout(
            hp.Choice(
                "dropout",
                values=[0.2,0.3,0.5]
            )
        ),

        tf.keras.layers.Dense(
            NUM_CLASSES,
            activation="softmax"
        )

    ])


    model.compile(
        optimizer=hp.Choice(
        "optimizer",
        values=["adam","rmsprop","sgd"]
    ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [18]:
tuner = kt.RandomSearch(
    build_resnet,
    objective="val_accuracy",
    max_trials=5,
    overwrite=True,
    directory="hyperparameter_tuning",
    project_name="resnet50"
)

In [19]:
tuner.search(
    train_dataset2,
    validation_data=val_dataset2,
    epochs=5,
    class_weight=class_weight_dict
)

Trial 5 Complete [00h 32m 15s]
val_accuracy: 0.6234198212623596

Best val_accuracy So Far: 0.6746506690979004
Total elapsed time: 02h 39m 30s


In [20]:
best_hps = tuner.get_best_hyperparameters(1)[0]

print(best_hps.values)

{'dense_units': 512, 'dropout': 0.5, 'optimizer': 'rmsprop'}


In [21]:
best_model = tuner.get_best_models(1)[0]

c:\Users\rizwa\.virtualenvs\week_7-vVhlwxiV\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(store)


In [22]:
history = best_model.fit(
    train_dataset2,
    validation_data=val_dataset2,
    epochs=5,
    class_weight=class_weight_dict,
    callbacks=[early_stop]
)

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 396s 2s/step - accuracy: 0.5687 - loss: 1.5185 - val_accuracy: 0.6720 - val_loss: 0.8839
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 390s 2s/step - accuracy: 0.5881 - loss: 1.3811 - val_accuracy: 0.6081 - val_loss: 1.0436
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 391s 2s/step - accuracy: 0.5814 - loss: 1.4447 - val_accuracy: 0.5156 - val_loss: 1.4057
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 393s 2s/step - accuracy: 0.5949 - loss: 1.3609 - val_accuracy: 0.6168 - val_loss: 1.0225
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 1.


In [23]:
test_loss, test_accuracy = best_model.evaluate(test_dataset2)

print("Test Accuracy :", test_accuracy)
print("Test Loss :", test_loss)

47/47 ━━━━━━━━━━━━━━━━━━━━ 70s 1s/step - accuracy: 0.6786 - loss: 0.8783
Test Accuracy : 0.6786426901817322
Test Loss : 0.8783023953437805


In [24]:
best_model.save(r'../models/resnet_hyper.keras')

In [25]:
history_df =pd.DataFrame(history.history)

history_df.to_csv(r'../log/res_net_hyper.csv',index=False)


In [ ]:
#optimizers 
#adam

In [26]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

resnet_model_adam = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),

    layers.Dropout(0.5),

    layers.Dense(7, activation="softmax")

])

resnet_model_adam.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,850,887 (90.98 MB)

 Trainable params: 263,175 (1.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [28]:
resnet_model_adam.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [29]:
resnet_model_adam.fit(train_dataset2,validation_data=val_dataset2,epochs= 5, class_weight=class_weight_dict,callbacks=[early_stop])

Epoch 1/5


c:\Users\rizwa\.virtualenvs\week_7-vVhlwxiV\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 396s 2s/step - accuracy: 0.3467 - loss: 1.7283 - val_accuracy: 0.4331 - val_loss: 1.4569
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 386s 2s/step - accuracy: 0.4523 - loss: 1.4156 - val_accuracy: 0.5875 - val_loss: 1.0559
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 388s 2s/step - accuracy: 0.4606 - loss: 1.3650 - val_accuracy: 0.5143 - val_loss: 1.1738
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 388s 2s/step - accuracy: 0.4822 - loss: 1.2757 - val_accuracy: 0.5988 - val_loss: 0.9815
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 388s 2s/step - accuracy: 0.5146 - loss: 1.2255 - val_accuracy: 0.5875 - val_loss: 1.0316
Restoring model weights from the end of the best epoch: 4.


In [30]:
resnet_model_adam.evaluate(test_dataset2)

47/47 ━━━━━━━━━━━━━━━━━━━━ 66s 1s/step - accuracy: 0.6201 - loss: 0.9677


[0.9676558375358582, 0.6200931668281555]

In [31]:
resnet_model_adam.save(r"../models/resenet_adam.keras")

In [ ]:
#sgd

In [32]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

resnet_model_sgd = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),

    layers.Dropout(0.5),

    layers.Dense(7, activation="softmax")

])

resnet_model_sgd.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,850,887 (90.98 MB)

 Trainable params: 263,175 (1.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [33]:
resnet_model_sgd.compile(optimizer="sgd",loss="sparse_categorical_crossentropy",metrics=["accuracy"])
resnet_model_sgd.fit(train_dataset2,validation_data=val_dataset2,epochs= 5, class_weight=class_weight_dict,callbacks=[early_stop])

Epoch 1/5


c:\Users\rizwa\.virtualenvs\week_7-vVhlwxiV\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 410s 2s/step - accuracy: 0.3199 - loss: 1.7373 - val_accuracy: 0.4604 - val_loss: 1.4740
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 402s 2s/step - accuracy: 0.4225 - loss: 1.4986 - val_accuracy: 0.5576 - val_loss: 1.1905
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 404s 2s/step - accuracy: 0.4678 - loss: 1.3796 - val_accuracy: 0.4032 - val_loss: 1.4557
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 403s 2s/step - accuracy: 0.4798 - loss: 1.3168 - val_accuracy: 0.4298 - val_loss: 1.6442
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 403s 2s/step - accuracy: 0.5059 - loss: 1.2791 - val_accuracy: 0.4438 - val_loss: 1.5493
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.


In [34]:
resnet_model_sgd.evaluate(test_dataset2)

47/47 ━━━━━━━━━━━━━━━━━━━━ 70s 1s/step - accuracy: 0.5749 - loss: 1.1770


[1.1770365238189697, 0.57485032081604]

In [35]:
resnet_model_sgd.save(r'../models/resenet_sgd.keras')

In [36]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

resnet_model_rms = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),

    layers.Dropout(0.5),

    layers.Dense(7, activation="softmax")

])

resnet_model_rms.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,850,887 (90.98 MB)

 Trainable params: 263,175 (1.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [37]:
resnet_model_rms.compile(optimizer=tf.keras.optimizers.RMSprop(),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
resnet_model_rms.fit(train_dataset2,validation_data=val_dataset2,epochs= 5, class_weight=class_weight_dict,callbacks=[early_stop])

Epoch 1/5


c:\Users\rizwa\.virtualenvs\week_7-vVhlwxiV\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 415s 2s/step - accuracy: 0.4606 - loss: 1.7552 - val_accuracy: 0.5775 - val_loss: 1.0592
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 405s 2s/step - accuracy: 0.5248 - loss: 1.4950 - val_accuracy: 0.6620 - val_loss: 0.8917
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 402s 2s/step - accuracy: 0.5352 - loss: 1.3707 - val_accuracy: 0.6687 - val_loss: 0.8412
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 396s 2s/step - accuracy: 0.5713 - loss: 1.3481 - val_accuracy: 0.6414 - val_loss: 0.9376
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 395s 2s/step - accuracy: 0.5764 - loss: 1.3270 - val_accuracy: 0.6507 - val_loss: 0.8939
Restoring model weights from the end of the best epoch: 3.


In [38]:
resnet_model_rms.evaluate(test_dataset2)

47/47 ━━━━━━━━━━━━━━━━━━━━ 65s 1s/step - accuracy: 0.6753 - loss: 0.8642


[0.8641818165779114, 0.6753160357475281]

In [39]:
resnet_model_rms.save(r'../models/resnet_rms.keras')
history_df =pd.DataFrame(history.history)

history_df.to_csv(r'../log/res_net_rms.csv',index=False)

In [ ]:
#64

In [47]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

resnet_model_64 = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),

    layers.Dropout(0.5),

    layers.Dense(7, activation="softmax")

])

resnet_model_64.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_4      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,850,887 (90.98 MB)

 Trainable params: 263,175 (1.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [50]:
resnet_model_64.compile(optimizer='adam',loss="sparse_categorical_crossentropy",metrics=["accuracy"])
resnet_model_64.fit(train_dataset_64,validation_data=val_dataset_64,epochs= 5, class_weight=class_weight_dict,callbacks=[early_stop])

Epoch 1/5


c:\Users\rizwa\.virtualenvs\week_7-vVhlwxiV\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


110/110 ━━━━━━━━━━━━━━━━━━━━ 403s 4s/step - accuracy: 0.3924 - loss: 1.6937 - val_accuracy: 0.5329 - val_loss: 1.2392
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 391s 4s/step - accuracy: 0.4691 - loss: 1.3898 - val_accuracy: 0.5882 - val_loss: 1.0465
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 390s 4s/step - accuracy: 0.4917 - loss: 1.3108 - val_accuracy: 0.5456 - val_loss: 1.1407
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 390s 4s/step - accuracy: 0.5138 - loss: 1.2339 - val_accuracy: 0.5349 - val_loss: 1.1643
Epoch 5/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 392s 4s/step - accuracy: 0.5299 - loss: 1.1743 - val_accuracy: 0.5768 - val_loss: 1.1060
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.


In [51]:
resnet_model_64.evaluate(test_dataset_64)

24/24 ━━━━━━━━━━━━━━━━━━━━ 68s 3s/step - accuracy: 0.5862 - loss: 1.0507


[1.0507138967514038, 0.5861610174179077]

In [52]:
resnet_model_rms.save(r'../models/resnet_64.keras')